In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('../dataset_builder/data/movies_dataset.csv')

In [4]:
print(df.columns , df.shape)

Index(['tmdb_id', 'imdb_id', 'title', 'release_date', 'genres', 'overview',
       'keywords', 'directors', 'cast', 'original_language', 'poster_url',
       'popularity', 'vote_average', 'vote_count'],
      dtype='str') (5880, 14)


In [5]:
df.isnull().sum()

tmdb_id                0
imdb_id                0
title                  0
release_date           0
genres                 1
overview               0
keywords             189
directors              0
cast                   1
original_language      0
poster_url             1
popularity             0
vote_average           0
vote_count             0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df = df[['title' , 'overview' , 'genres' , 'keywords' , 'cast' , 'directors']]
df.head(2)

,title,overview,genres,keywords,cast,directors
0,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","Drama, Crime","based on novel or book, loss of loved one, lov...","Marlon Brando, Al Pacino, James Caan, Robert D...",Francis Ford Coppola
1,Star Wars,Princess Leia is captured and held hostage by ...,"Adventure, Action, Science Fiction","empire, galaxy, rebellion, android, hermit, sm...","Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas


In [8]:
df.isnull().sum()

title          0
overview       0
genres         1
keywords     189
cast           1
directors      0
dtype: int64

In [9]:
df['genres'] = df['genres'].fillna('')
df['keywords'] = df['keywords'].fillna('')
df['cast'] = df['cast'].fillna('')

In [10]:
df.isnull().sum()

title        0
overview     0
genres       0
keywords     0
cast         0
directors    0
dtype: int64

In [11]:
df['combined_feature'] = df['overview'] + " " + df['genres'] + " " + df['keywords'] + " " + df['cast'] + " " + df['directors']

In [12]:
df.head(2)

,title,overview,genres,keywords,cast,directors,combined_feature
0,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","Drama, Crime","based on novel or book, loss of loved one, lov...","Marlon Brando, Al Pacino, James Caan, Robert D...",Francis Ford Coppola,"Spanning the years 1945 to 1955, a chronicle o..."
1,Star Wars,Princess Leia is captured and held hostage by ...,"Adventure, Action, Science Fiction","empire, galaxy, rebellion, android, hermit, sm...","Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas,Princess Leia is captured and held hostage by ...


In [13]:
df['combined_feature'][0]

'Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito Corleone barely survives an attempt on his life, his youngest son, Michael steps in to take care of the would-be killers, launching a campaign of bloody revenge. Drama, Crime based on novel or book, loss of loved one, love at first sight, italy, gangster, symbolism, patriarch, europe, organized crime, mafia Marlon Brando, Al Pacino, James Caan, Robert Duvall, Richard S. Castellano, Diane Keaton, Talia Shire, Gianni Russo, Sterling Hayden, John Marley Francis Ford Coppola'

In [14]:
# lemmitization : converting each word to its root form ; end  , ending --> end

In [15]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [16]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\KIIT0001\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\KIIT0001\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [17]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [18]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '' , text)

    words = text.split()
    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [19]:
df['combined_feature'] = df['combined_feature'].apply(preprocess_text)

In [20]:
df['combined_feature'][0]

'spanning year chronicle fictional italianamerican corleone crime family organized crime family patriarch vito corleone barely survives attempt life youngest son michael step take care wouldbe killer launching campaign bloody revenge drama crime based novel book loss loved one love first sight italy gangster symbolism patriarch europe organized crime mafia marlon brando al pacino james caan robert duvall richard castellano diane keaton talia shire gianni russo sterling hayden john marley francis ford coppola'

In [21]:
df.duplicated().sum()

np.int64(0)

In [22]:
indices = pd.Series(df.index , index = df['title'])
indices

title
The Godfather                 0
Star Wars                     1
The Godfather Part II         2
Scarface                      3
Alien                         4
                           ... 
Iqbal                      5875
Manorama Six Feet Under    5876
Tere Bin Laden             5877
Phas Gaye Re Obama         5878
Himmatwala                 5879
Length: 5880, dtype: int64

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [24]:
tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1,2) , stop_words='english')

In [25]:
tfidf_matrix = tfidf.fit_transform(df['combined_feature'])

In [25]:
# print(tfidf_matrix[0])

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def recommend(title, n=10):
    if title not in indices:
        return ['Movie not found']
    
    idx = indices[title]
    if isinstance(idx , pd.Series):
        idx = idx.iloc[0]
    sim_score = cosine_similarity(tfidf_matrix[idx] , tfidf_matrix).flatten()
    similar_idx = sim_score.argsort()[::-1][0:n+1]
    return df['title'].iloc[similar_idx]

In [97]:
# recommend('The Avengers')
df['title'].duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
5875    False
5876    False
5877    False
5878    False
5879    False
Name: title, Length: 5880, dtype: bool

In [93]:
import pickle

pickle.dump(tfidf_matrix, open('tfidf_matrix.pkl' , 'wb'))
pickle.dump(indices,open('indices.pkl', 'wb'))
df.to_pickle('df.pkl')
pickle.dump(tfidf, open('tfidf.pkl', 'wb'))